# 🚕 Taxi Trip Fare Prediction

> **Machine Learning Regression Project**

---

## 📌 1. Project Overview

This project focuses on predicting **taxi trip fares** using Machine Learning regression techniques.

The goal is to build a model that can estimate the fare of a taxi trip based on information such as trip characteristics, location, and time.

---

## 🎯 2. Problem Statement

Given information about a taxi trip, the objective is to **predict the fare amount** of that trip.

### Target Variable

**Fare Amount**

### Machine Learning Task

**Regression**

---

## 🔍 3. Project Objectives

Throughout this project, we will:

- Understand and explore the dataset.
- Identify and handle data quality issues.
- Perform Exploratory Data Analysis (EDA).
- Engineer meaningful features from the raw data.
- Prepare the data for Machine Learning.
- Train and compare multiple regression models.
- Evaluate model performance using appropriate regression metrics.
- Analyze prediction errors and model limitations.
- Save the final trained model for future use.

---

## 🛠️ 4. Project Workflow

```text
Data Understanding
        ↓
Exploratory Data Analysis
        ↓
Data Cleaning
        ↓
Feature Engineering
        ↓
Data Preprocessing
        ↓
Model Training
        ↓
Model Evaluation
        ↓
Error Analysis
        ↓
Model Saving
```

---

### 📚 Skills & Concepts

**Regression • EDA • Data Cleaning • Feature Engineering • Model Evaluation • Error Analysis • Machine Learning**

In [ ]:
import sys
from pathlib import Path
import requests
import pandas as pd

print(sys.executable)

In [ ]:
raw_path = Path("../data/raw")
raw_path.mkdir(parents=True, exist_ok=True)

csv_path = raw_path / "nyc_taxi_fare_sample.csv"

In [ ]:
url = "https://data.cityofnewyork.us/resource/gkne-dk5s.json"

params = {
    "$limit": 100_000
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()

df = pd.DataFrame(data)

print("Dataset extracted successfully!")
print(f"Shape: {df.shape}")

In [ ]:
df.to_csv(csv_path, index=False)

print(f"Saved to: {csv_path}")

## 1. Data Understanding & Data Quality

Before building any Machine Learning model, it is important to understand the structure and quality of the dataset.

In this section, we will:

- Inspect the first and last records.
- Check the dataset dimensions.
- Examine the column names and data types.
- Identify missing values.
- Check for duplicate records.
- Explore basic statistical properties.

The goal is to understand the raw dataset and identify potential data quality issues before performing data cleaning and feature engineering.

### 1.1 Dataset Overview

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns

 ### 1.2 Data Types

In [ ]:
df.info()

### 1.3 Missing Values

In [ ]:
missing_values = df.isnull().sum()

missing_values[missing_values > 0]

### 1.4 Duplicate Records

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

### Key Observations

- The dataset contains 100,000 records and 18 columns.
- All columns are currently stored as strings.
- `store_and_fwd_flag` contains 51,032 missing values.
- No completely duplicated rows were found.
- Numerical statistics cannot be properly interpreted until the data types are converted.

## 2. Data Cleaning

In this section, we will correct data types, handle missing values, and identify invalid or inconsistent values before preparing the data for modeling.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import datetime
import pandas as pd

 ### 2.1 Convert Data Types

In [ ]:
# Datetime columns
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df["dropoff_datetime"] = pd.to_datetime(df["dropoff_datetime"])

# Numerical columns
numeric_columns = [
    "passenger_count",
    "trip_distance",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "fare_amount",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "imp_surcharge",
    "rate_code"
]

df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

In [ ]:
df.info()

**Observation:** Data types have been successfully converted to appropriate formats. `store_and_fwd_flag` remains the only column with missing values.

 ### 2.2 Handle Missing Values

In [ ]:
df.isnull().sum()

Since `store_and_fwd_flag` contains a large proportion of missing values, the missing entries will be treated as an `Unknown` category rather than removing the corresponding records.

In [ ]:
df["store_and_fwd_flag"] = df["store_and_fwd_flag"].fillna("Unknown")

In [ ]:
df["store_and_fwd_flag"].value_counts(dropna=False)

 ### 2.3 Invalid Values

In [ ]:
df["passenger_count"].describe()

In [ ]:
df["trip_distance"].describe()

In [ ]:
df[df["passenger_count"] == 0][["passenger_count", "trip_distance", "fare_amount"]].head()

In [ ]:
df[df["trip_distance"] == 0][["passenger_count", "trip_distance", "fare_amount"]].head()

In [ ]:
# Remove records with invalid passenger count
df = df[df["passenger_count"] > 0].copy()

print(f"Shape after removing invalid passenger counts: {df.shape}")

In [ ]:
df["fare_amount"].describe()

In [ ]:
df[df["fare_amount"] <= 0][
    ["passenger_count", "trip_distance", "fare_amount"]
]

In [ ]:
# Remove records with non-positive fare
df = df[df["fare_amount"] > 0].copy()

print(f"Shape after removing invalid fares: {df.shape}")

### 2.4 Coordinate Validation

In [ ]:
df[[
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude"
]].describe()

In [ ]:
coordinate_columns = [
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude"
]

(df[coordinate_columns] == 0).sum()

In [ ]:
invalid_coordinates = (
    (df["pickup_longitude"] == 0) |
    (df["pickup_latitude"] == 0) |
    (df["dropoff_longitude"] == 0) |
    (df["dropoff_latitude"] == 0)
)

print(f"Invalid coordinate records: {invalid_coordinates.sum()}")

In [ ]:
# Remove records with invalid coordinates
df = df[~invalid_coordinates].copy()

print(f"Shape after removing invalid coordinates: {df.shape}")

**Observation:** 2,217 records with invalid coordinates were removed, leaving only trips with valid pickup and drop-off locations.

 ### 2.5 Trip Duration Validation

In [ ]:
df["trip_duration"] = (
    df["dropoff_datetime"] - df["pickup_datetime"]
).dt.total_seconds() / 60

df["trip_duration"].describe()

In [ ]:
(df["trip_duration"] < 0).sum()

In [ ]:
df["trip_duration"].isnull().sum()

In [ ]:
# Remove records with invalid trip duration
df = df[df["trip_duration"] >= 0].copy()

print(f"Final shape after data cleaning: {df.shape}")

**Observation:** Invalid records were removed based on passenger count, fare amount, coordinates, and trip duration. Extreme but potentially valid values were retained for further outlier analysis during EDA.

## 3. Exploratory Data Analysis (EDA)

We explore the cleaned dataset to understand the distributions, relationships, and potential outliers before feature engineering and modeling.

### 3.1 Numerical Summary

In [ ]:
df.describe()

### Key Observations

- The cleaned dataset contains 97,836 trips.
- `fare_amount`, `trip_distance`, and `trip_duration` show potentially extreme values that require further investigation.
- The target variable appears right-skewed, with the mean fare higher than the median.
- Some geographic values are unusually distant from the main NYC coordinate range.
- Further outlier analysis will be performed before modeling.

### 3.2 Target Variable Distributio

In [ ]:
df["fare_amount"].hist(bins=50, figsize=(10, 5))

plt.xlabel("Fare Amount")
plt.ylabel("Frequency")
plt.title("Distribution of Fare Amount")
plt.show()

**Observation:** `fare_amount` is strongly right-skewed. Most trips have relatively low fares, while a small number of trips have much higher fares that require further outlier analysis.

### 3.3 Feature Distributions

In [ ]:
df["trip_distance"].hist(bins=50, figsize=(10, 5))

plt.xlabel("Trip Distance")
plt.ylabel("Frequency")
plt.title("Distribution of Trip Distance")
plt.show()

**Observation:** `trip_distance` is strongly right-skewed. Most trips are relatively short, while a small number of trips have much larger distances that require further outlier analysis.

In [ ]:
plt.figure(figsize=(10, 5))

plt.scatter(
    df["trip_distance"],
    df["fare_amount"],
    alpha=0.3
)

plt.xlabel("Trip Distance")
plt.ylabel("Fare Amount")
plt.title("Trip Distance vs Fare Amount")

plt.show()

**Observation:** `trip_distance` shows a clear positive relationship with `fare_amount`. In general, longer trips tend to have higher fares, although some extreme observations are present and will be investigated during outlier analysis.

### 3.5 Outlier Analysis

In [ ]:
Q1 = df["fare_amount"].quantile(0.25)
Q3 = df["fare_amount"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1}")
print(f"Q3: {Q3}")
print(f"IQR: {IQR}")
print(f"Lower Bound: {lower_bound}")
print(f"Upper Bound: {upper_bound}")

In [ ]:
fare_outliers = df[df["fare_amount"] > upper_bound]

print(f"Number of potential outliers: {len(fare_outliers)}")
print(f"Percentage of potential outliers: {len(fare_outliers) / len(df) * 100:.2f}%")

**Observation:** The IQR method identifies 7,689 potential fare outliers (7.87% of the dataset). Since this is a substantial proportion, these observations will be investigated rather than removed automatically.

In [ ]:
Q1_distance = df["trip_distance"].quantile(0.25)
Q3_distance = df["trip_distance"].quantile(0.75)

IQR_distance = Q3_distance - Q1_distance

lower_bound_distance = Q1_distance - 1.5 * IQR_distance
upper_bound_distance = Q3_distance + 1.5 * IQR_distance

print(f"Q1: {Q1_distance}")
print(f"Q3: {Q3_distance}")
print(f"IQR: {IQR_distance}")
print(f"Lower Bound: {lower_bound_distance}")
print(f"Upper Bound: {upper_bound_distance}")

In [ ]:
distance_outliers = df[df["trip_distance"] > upper_bound_distance]

print(f"Number of potential outliers: {len(distance_outliers)}")
print(f"Percentage of potential outliers: {len(distance_outliers) / len(df) * 100:.2f}%")

**Observation:** The IQR method identifies 9,768 potential trip-distance outliers (9.98% of the dataset). Since this represents a substantial proportion of the data, these observations will be retained for further analysis rather than removed automatically.

## 4. Feature Engineering

We transform raw trip information into meaningful features that can help the model predict taxi fares.

### 4.1 Datetime Features

We extract useful time-based features from `pickup_datetime`, such as hour, day, month, and day of the week.

In [ ]:
df["pickup_hour"] = df["pickup_datetime"].dt.hour
df["pickup_day"] = df["pickup_datetime"].dt.day
df["pickup_month"] = df["pickup_datetime"].dt.month
df["pickup_day_of_week"] = df["pickup_datetime"].dt.dayofweek

df["pickup_is_weekend"] = (
    df["pickup_day_of_week"].isin([5, 6]).astype(int)
)

In [ ]:
df[
    [
        "pickup_datetime",
        "pickup_hour",
        "pickup_day",
        "pickup_month",
        "pickup_day_of_week",
        "pickup_is_weekend"
    ]
].head()

In [ ]:
df["is_rush_hour"] = (
    df["pickup_hour"].isin([7, 8, 9, 16, 17, 18, 19])
).astype(int)

In [ ]:
df["is_rush_hour"].value_counts()

In [ ]:
df["is_rush_hour"].value_counts(normalize=True) * 100

In [ ]:
pd.crosstab(
    df["pickup_hour"],
    df["is_rush_hour"]
)

### 4.2 Geospatial Features

In [ ]:
import numpy as np

In [ ]:
lat1 = np.radians(df["pickup_latitude"])
lon1 = np.radians(df["pickup_longitude"])

lat2 = np.radians(df["dropoff_latitude"])
lon2 = np.radians(df["dropoff_longitude"])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
)

c = 2 * np.arcsin(np.sqrt(a))

earth_radius_km = 6371

df["geodesic_distance_km"] = earth_radius_km * c

In [ ]:
df["geodesic_distance_km"].describe()

In [ ]:
df[df["geodesic_distance_km"] > 100][
    [
        "pickup_latitude",
        "pickup_longitude",
        "dropoff_latitude",
        "dropoff_longitude",
        "geodesic_distance_km",
        "fare_amount"
    ]
].head()

In [ ]:
(df["geodesic_distance_km"] > 100).sum()

In [ ]:
df = df[df["geodesic_distance_km"] <= 100].copy()

In [ ]:
df["geodesic_distance_km"].describe()

In [ ]:
df["is_same_location"] = (
    df["geodesic_distance_km"] < 0.1
).astype(int)

In [ ]:
df["is_same_location"].value_counts()

**Observation:** Geospatial features were created using the pickup and dropoff coordinates. A small number of trips have pickup and dropoff locations within 100 meters of each other.

### 4.3 Categorical Features

Categorical features are converted into numerical representations so they can be used by machine learning models.

In [ ]:
df[["vendor_id", "payment_type", "store_and_fwd_flag"]].nunique()

In [ ]:
for col in ["vendor_id", "payment_type", "store_and_fwd_flag"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

### 4.4 Feature Selection

We select features for predicting `fare_amount` and exclude direct fare components such as `total_amount`, `tip_amount`, and `tolls_amount`. **Prediction timing matters:** `payment_type`, `trip_distance`, and drop-off coordinates may only be known during or after a trip. If the goal is to predict the fare before a trip starts, remove features that are not available at pickup time.


In [ ]:
target = "fare_amount"

features = [
    "vendor_id",
    "passenger_count",
    "trip_distance",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "store_and_fwd_flag",
    "payment_type",
    "rate_code",
    "pickup_hour",
    "pickup_day",
    "pickup_month",
    "pickup_day_of_week",
    "pickup_is_weekend",
    "geodesic_distance_km",
    "is_same_location"
]

X = df[features]
y = df[target]

In [ ]:
X.shape, y.shape

### 4.5 Train-Test Split

The dataset is split into training and testing sets to train the model and evaluate its performance on unseen data.

In [ ]:
%pip install scikit-learn

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

### 4.6 Encoding Categorical Features

Categorical features are encoded using One-Hot Encoding so they can be used by machine learning models.

In [ ]:
categorical_features = [
    "vendor_id",
    "store_and_fwd_flag",
    "payment_type",
    "rate_code"
]

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

## 5. Model Building

We train machine learning models to predict `fare_amount`.

### 5.1 Linear Regression

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

In [ ]:
linear_model.fit(X_train, y_train)

In [ ]:
y_pred_lr = linear_model.predict(X_test)

In [ ]:
print(y_pred_lr[:10])

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred_lr
})

comparison.head(10)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred_lr)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2 = r2_score(y_test, y_pred_lr)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

**Observation:** The Linear Regression model achieved an MAE of 1.77, RMSE of 3.00, and R² of 0.913. This provides a strong baseline for comparing more advanced models.

### 5.2 Random Forest

We use Random Forest Regressor as a non-linear model to compare its performance with Linear Regression.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [ ]:
rf_model.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test)

In [ ]:
print(y_pred_rf[:10])

### 5.3 Scikit-learn Model Comparison

The Scikit-learn Linear Regression and Random Forest models are compared using MAE, RMSE, and R² on the same unseen test set. A broader comparison including the from-scratch implementations appears in Section 6.4.


In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred_rf
})

comparison.head(10)

In [ ]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R²:", r2_rf)

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [mae, mae_rf],
    "RMSE": [rmse, rmse_rf],
    "R²": [r2, r2_rf]
})

results

**Observation:** Random Forest achieved lower MAE and RMSE and a higher R² than Linear Regression on the test set.

## 6. Model Building From Scratch

### 6.1 Linear Regression From Scratch


#### 6.1.1 Data Preparation

The data is prepared by encoding categorical features and converting the datasets into NumPy arrays.


In [ ]:
X_train_scratch = pd.get_dummies(
    X_train,
    columns=categorical_features,
    dtype=int
)

X_test_scratch = pd.get_dummies(
    X_test,
    columns=categorical_features,
    dtype=int
)

In [ ]:
X_train_scratch, X_test_scratch = X_train_scratch.align(
    X_test_scratch,
    join="left",
    axis=1,
    fill_value=0
)

In [ ]:
X_train_scratch = X_train_scratch.values
X_test_scratch = X_test_scratch.values

y_train_scratch = y_train.values
y_test_scratch = y_test.values

In [ ]:
print("X_train:", X_train_scratch.shape)
print("X_test:", X_test_scratch.shape)
print("y_train:", y_train_scratch.shape)
print("y_test:", y_test_scratch.shape)

### 6.1.2 Model Implementation

In [ ]:
import numpy as np

X_train_scratch = np.c_[
    np.ones(X_train_scratch.shape[0]),
    X_train_scratch
]

X_test_scratch = np.c_[
    np.ones(X_test_scratch.shape[0]),
    X_test_scratch
]

In [ ]:
weights = np.zeros(X_train_scratch.shape[1])

In [ ]:
print("Number of weights:", len(weights))

In [ ]:
def predict(X, weights):
    return X @ weights

#### 6.1.3 Cost Function

The Mean Squared Error is used as the cost function to measure the difference between predicted and actual fare values.

In [ ]:
def compute_cost(X, y, weights):
    predictions = predict(X, weights)
    errors = predictions - y
    
    cost = np.mean(errors ** 2) / 2
    
    return cost

#### 6.1.4 Feature Scaling

Feature scaling is applied to numerical features to keep their values on a similar scale and improve the stability of Gradient Descent.

In [ ]:
X_mean = X_train_scratch[:, 1:].mean(axis=0)
X_std = X_train_scratch[:, 1:].std(axis=0)

X_std[X_std == 0] = 1

X_train_scaled = X_train_scratch.copy()
X_test_scaled = X_test_scratch.copy()

X_train_scaled[:, 1:] = (
    X_train_scratch[:, 1:] - X_mean
) / X_std

X_test_scaled[:, 1:] = (
    X_test_scratch[:, 1:] - X_mean
) / X_std

#### 6.1.5 Gradient Descent

Gradient Descent is used to update the model weights and minimize the cost function during training.

In [ ]:
def train_linear_regression(X, y, learning_rate=0.01, epochs=1000):
    weights = np.zeros(X.shape[1])
    
    for _ in range(epochs):
        predictions = predict(X, weights)
        errors = predictions - y
        
        gradients = (X.T @ errors) / len(y)
        
        weights -= learning_rate * gradients
    
    return weights

In [ ]:
weights = train_linear_regression(
    X_train_scaled,
    y_train_scratch,
    learning_rate=0.01,
    epochs=1000
)

In [ ]:
print(weights[:10])

#### 6.1.5 Prediction

The trained model is used to predict taxi fares on the test set.

In [ ]:
y_pred_scratch = predict(
    X_test_scaled,
    weights
)

In [ ]:
print(y_pred_scratch[:10])

In [ ]:
comparison_scratch = pd.DataFrame({
    "Actual": y_test_scratch,
    "Predicted": y_pred_scratch
})

comparison_scratch.head(10)

In [ ]:
mae_scratch = mean_absolute_error(
    y_test_scratch,
    y_pred_scratch
)

rmse_scratch = np.sqrt(
    mean_squared_error(
        y_test_scratch,
        y_pred_scratch
    )
)

r2_scratch = r2_score(
    y_test_scratch,
    y_pred_scratch
)

print("MAE:", mae_scratch)
print("RMSE:", rmse_scratch)
print("R²:", r2_scratch)

In [ ]:
comparison_models = pd.DataFrame({
    "Model": [
        "Linear Regression - Sklearn",
        "Linear Regression - From Scratch"
    ],
    "MAE": [
        mae,
        mae_scratch
    ],
    "RMSE": [
        rmse,
        rmse_scratch
    ],
    "R²": [
        r2,
        r2_scratch
    ]
})

comparison_models

**Observation:** The Linear Regression model implemented from scratch achieved performance close to the Scikit-learn implementation, with slightly higher MAE and RMSE and a slightly lower R².

### 6.2 Decision Tree From Scratch

A Decision Tree Regressor is implemented from scratch. Candidate splits are evaluated using Mean Squared Error (MSE). To keep runtime manageable, the tree is trained on a reproducible sample of the training data and evaluated on the full unseen test set.


#### 6.2.1 Decision Tree Implementation

The tree recursively selects the split with the lowest weighted MSE. A maximum number of candidate thresholds is used for numeric features to reduce computation.

In [ ]:
def mse(y):
    if len(y) == 0:
        return 0.0
    return np.mean((y - np.mean(y)) ** 2)


def find_best_split(X, y, max_features=None, max_thresholds=10, rng=None):
    if rng is None:
        rng = np.random.default_rng(42)

    n_samples, n_features = X.shape
    if max_features is None or max_features >= n_features:
        feature_indices = np.arange(n_features)
    else:
        feature_indices = rng.choice(n_features, size=max_features, replace=False)

    best_feature, best_threshold = None, None
    best_error = float("inf")

    for feature in feature_indices:
        values = np.unique(X[:, feature])
        if len(values) < 2:
            continue

        if len(values) > max_thresholds + 1:
            thresholds = np.unique(np.quantile(
                values, np.linspace(0.1, 0.9, max_thresholds)
            ))
        else:
            thresholds = (values[:-1] + values[1:]) / 2

        for threshold in thresholds:
            left_mask = X[:, feature] <= threshold
            right_mask = ~left_mask
            n_left, n_right = left_mask.sum(), right_mask.sum()
            if n_left == 0 or n_right == 0:
                continue

            weighted_error = (
                n_left * mse(y[left_mask]) + n_right * mse(y[right_mask])
            ) / n_samples

            if weighted_error < best_error:
                best_error = weighted_error
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold


def build_tree(
    X, y, depth=0, max_depth=8, min_samples_split=30,
    max_features=None, max_thresholds=10, rng=None
):
    if rng is None:
        rng = np.random.default_rng(42)

    if (depth >= max_depth or len(y) < min_samples_split or
            np.all(y == y[0])):
        return {"value": float(np.mean(y))}

    feature, threshold = find_best_split(
        X, y, max_features=max_features,
        max_thresholds=max_thresholds, rng=rng
    )
    if feature is None:
        return {"value": float(np.mean(y))}

    left_mask = X[:, feature] <= threshold
    right_mask = ~left_mask
    if left_mask.sum() == 0 or right_mask.sum() == 0:
        return {"value": float(np.mean(y))}

    return {
        "feature": int(feature),
        "threshold": float(threshold),
        "left": build_tree(
            X[left_mask], y[left_mask], depth + 1, max_depth,
            min_samples_split, max_features, max_thresholds, rng
        ),
        "right": build_tree(
            X[right_mask], y[right_mask], depth + 1, max_depth,
            min_samples_split, max_features, max_thresholds, rng
        )
    }


def predict_tree(tree, X):
    predictions = np.empty(X.shape[0], dtype=float)
    for i, row in enumerate(X):
        node = tree
        while "value" not in node:
            if row[node["feature"]] <= node["threshold"]:
                node = node["left"]
            else:
                node = node["right"]
        predictions[i] = node["value"]
    return predictions


#### 6.2.2 Training and Evaluating the Decision Tree on Unseen Data

The first column of `X_train_scratch` and `X_test_scratch` is the bias column added for Linear Regression. It is removed here because tree models do not need a bias column. The test set remains untouched during training.

In [ ]:
# Remove the bias column used by the scratch Linear Regression model
X_train_tree_all = X_train_scratch[:, 1:]
X_test_tree = X_test_scratch[:, 1:]

# Use a reproducible subset of the training set to reduce runtime
sample_size_tree = min(10000, len(y_train_scratch))
rng_tree = np.random.default_rng(42)
tree_indices = rng_tree.choice(
    len(y_train_scratch), size=sample_size_tree, replace=False
)
X_tree_train = X_train_tree_all[tree_indices]
y_tree_train = y_train_scratch[tree_indices]

decision_tree = build_tree(
    X_tree_train, y_tree_train,
    max_depth=8,
    min_samples_split=30,
    max_features=None,
    max_thresholds=10,
    rng=np.random.default_rng(42)
)

y_pred_tree = predict_tree(decision_tree, X_test_tree)

mae_tree = mean_absolute_error(y_test_scratch, y_pred_tree)
rmse_tree = np.sqrt(mean_squared_error(y_test_scratch, y_pred_tree))
r2_tree = r2_score(y_test_scratch, y_pred_tree)

print("Decision Tree - From Scratch (Unseen Test Data)")
print(f"MAE:  {mae_tree:.4f}")
print(f"RMSE: {rmse_tree:.4f}")
print(f"R²:   {r2_tree:.4f}")


### 6.3 Random Forest From Scratch

A Random Forest combines multiple decision trees. Each tree is trained on a bootstrap sample, and each split considers a random subset of features. The forest is evaluated on the same unseen test set.

In [ ]:
def bootstrap_sample(X, y, rng):
    indices = rng.choice(len(y), size=len(y), replace=True)
    return X[indices], y[indices]


def build_random_forest(
    X, y, n_trees=10, max_depth=8, min_samples_split=30,
    max_thresholds=10, random_state=42
):
    rng = np.random.default_rng(random_state)
    max_features = max(1, int(np.sqrt(X.shape[1])))
    forest = []

    for i in range(n_trees):
        X_boot, y_boot = bootstrap_sample(X, y, rng)
        tree_rng = np.random.default_rng(rng.integers(0, 2**32 - 1))
        tree = build_tree(
            X_boot, y_boot,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            max_features=max_features,
            max_thresholds=max_thresholds,
            rng=tree_rng
        )
        forest.append(tree)
        print(f"Tree {i + 1}/{n_trees} built")

    return forest


def predict_forest(forest, X):
    tree_predictions = np.array([predict_tree(tree, X) for tree in forest])
    return tree_predictions.mean(axis=0)


#### 6.3.1 Training and Evaluating the Random Forest on Unseen Data

The forest uses the same training subset size policy as the from-scratch Decision Tree. Predictions are evaluated against the full unseen test set.

In [ ]:
# Reuse the reproducible 10,000-row training sample
forest = build_random_forest(
    X_tree_train,
    y_tree_train,
    n_trees=10,
    max_depth=8,
    min_samples_split=30,
    max_thresholds=10,
    random_state=42
)

y_pred_rf_scratch = predict_forest(forest, X_test_tree)

mae_rf_scratch = mean_absolute_error(y_test_scratch, y_pred_rf_scratch)
rmse_rf_scratch = np.sqrt(
    mean_squared_error(y_test_scratch, y_pred_rf_scratch)
)
r2_rf_scratch = r2_score(y_test_scratch, y_pred_rf_scratch)

print("Random Forest - From Scratch (Unseen Test Data)")
print(f"MAE:  {mae_rf_scratch:.4f}")
print(f"RMSE: {rmse_rf_scratch:.4f}")
print(f"R²:   {r2_rf_scratch:.4f}")


### 6.4 Final Model Comparison on Unseen Test Data

All models below are evaluated on the same held-out test set. The from-scratch tree models use a smaller training sample to reduce runtime, so their scores should be interpreted with that difference in mind.

In [ ]:
# Scikit-learn predictions on the held-out test set
# y_pred_lr and y_pred_rf were produced by the fitted pipelines above.
model_predictions = {
    "Linear Regression - Scikit-learn": y_pred_lr,
    "Random Forest - Scikit-learn": y_pred_rf,
    "Linear Regression - From Scratch": y_pred_scratch,
    "Decision Tree - From Scratch": y_pred_tree,
    "Random Forest - From Scratch": y_pred_rf_scratch,
}

comparison_rows = []
for model_name, predictions in model_predictions.items():
    comparison_rows.append({
        "Model": model_name,
        "MAE": mean_absolute_error(y_test_scratch, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test_scratch, predictions)),
        "R²": r2_score(y_test_scratch, predictions),
    })

comparison_models = pd.DataFrame(comparison_rows).sort_values(
    "RMSE", ascending=True
).reset_index(drop=True)
comparison_models.round(4)


In [ ]:
import matplotlib.pyplot as plt

metrics = [
    ("MAE", "MAE (Lower is Better)"),
    ("RMSE", "RMSE (Lower is Better)"),
    ("R²", "R² (Higher is Better)"),
]

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for ax, (metric, title) in zip(axes, metrics):
    bars = ax.bar(comparison_models["Model"], comparison_models[metric])
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=75)
    ax.bar_label(bars, fmt="%.3f", padding=3)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.suptitle("Model Comparison — Unseen Test Data")
plt.tight_layout()
plt.show()


## 8. Conclusion

In this project, five regression models were implemented and evaluated for NYC taxi fare prediction: Linear Regression and Random Forest using Scikit-learn, along with Linear Regression, Decision Tree, and Random Forest implemented from scratch.

The Scikit-learn Random Forest model achieved the best overall performance, with an MAE of 1.2825, an RMSE of 2.9122, and an R² score of 0.9262. These results indicate that the model performed better than the other evaluated models on the test set.

The results also demonstrate the value of comparing different regression algorithms and implementing machine learning models from scratch to understand their underlying mechanisms.

Overall, this project provided practical experience in data cleaning, feature engineering, model implementation, and regression evaluation. Further improvements could include hyperparameter tuning, more robust outlier analysis, and ensuring that all prediction features are available at the intended prediction time.